# Deliverable 2
Group G4_2: Nikolai Enger Hoff & Tanja Hasdal Gundersen \
Github link: https://github.com/NickNack124/INF201-group-deliverable-2

## Imports
We start by importing the modules that will be used for this asisgnment.

In [20]:
import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


## Task 1
The goal of this task is to define functions we can use to read a file and sort its contents into dictionaries, which can then be used later for analytical purposes.

In [ ]:
def file_reader(file: Path) -> tuple[str, str]:
    """Function that reads a file using using either utf-8 or latin1 as its encoding language."""

    try:
        # We first attempt to encode using utf-8
        encoder = 'utf-8'
        file_text_content = file.open(encoding=encoder).readlines()

    except UnicodeEncodeError:
        # In case utf-8 fails, we attempt to encode with latin1
        encoder = 'latin1'
        file_text_content = file.open(encoding=encoder).readlines()

    return (''.join(file_text_content), encoder)


def log_parser(file: Path) -> list[dict]:
    """Function that parses the output from the file_reader as a list of dictionaries."""

    # We read the path object with the use of the previous function
    file_details: tuple[str, str] = file_reader(file)

    # Defining a regex pattern
    pattern = r"""
        \[(?P<timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\] # The time of the transaction
        \s+\[(?P<register_id>\w{3}_\d{2})\]                     # The register the transaction occured at
        \s+item=(?P<item>\w+)                                   # The item that was bought
        \s+price=(?P<price>\d+\.\d+)                            # The price of the item
        \s+(?P<currency>\w+)                                    # The currency the customer paid in
        \s+\((?P<status>\w+)\)                                  # The status of the transaction
    """
    # Making a dict based on the pattern
    transactions = [m.groupdict() for m in re.finditer(pattern, file_details[0], re.VERBOSE)]
    # Verbose allows us to keep the pattern neat, using docstring and comments

    # We then convert the price to a floating point number
    for record in transactions:
        record["price"] = float(record["price"])

    return transactions


The functions were tested on several inputs, and it did work as intended.

## Task 2
In this task, the goal is to find the files within the "data" folder (for further information, see README.md), as well as analyzing the metadata file also contained within.

In [ ]:
def file_finder(path: str | Path, suffix: str = '.log') -> list[Path]:
    """Returns all the files of a specific suffix in a directory as paths in a list."""

    # First we convert the input to a valid path object
    dir = Path(path)
    if not dir.exists():
        dir = Path('..') / path
        # Allows the code to work as long as the input object at least shares a parent directory with the code

    # We then recursively dig through the directory and its subdirectories
    filelist: list[Path] = []
    for f in dir.iterdir():
        if f.is_dir():
            # We recursively call the function again, and extend filelist by the returned filelist
            filelist.extend(file_finder(str(f), suffix=suffix))

        elif f.is_file() and f.suffix == suffix:
            # If the file matches the suffix (default: '.log'), it gets appended
            filelist.append(f)

    return filelist


def metadata_reader(source: list | dict | str) -> set[str]:
    """This function reads a json dictionary and returns a set of the metadata."""

    # first we extract the json dictionary from the path
    if type(source) is str:
        source = json.load(file_finder(Path(source), suffix='.json')[0].open())
        # This does rely on the desired file being the first/only json-file in the directory

    # We standardize the iterable so the function can handle both lists and dictionaries
    if type(source) is dict:
        metadata = source.items()
    else:
        metadata = [(k, v) for k, v in enumerate(source)]

    # Recursive loop for updating the set of active_registers
    active_registers: set[str] = set()
    for k, v in metadata:

        if type(v) in (list, dict):
            # updates the set using the returned set (the functions final return)
            active_registers.update(metadata_reader(v))

        # Adds v if it is the boolean value True and its key is a variation of "active"
        elif type(k) is str and k.lower() == "active" and type(v) is bool and v:
            # We make sure it actually has an "active" key, in case this dictionary were to ever include other booleans
            # This does still require the dict-items ("id", str) and (str, bool) to be in source
            active_registers.add(source["id"])   # type: ignore (the logic gate already gatekeeps this part to dict-sources)

    return active_registers


The functions were tested on several inputs, and it did work as intended.

## Task 3
We want to remove glitch ouliers in the prices, as wall as turn the prices into a 1D array. We also want to calculate the refund ratio using the refund count and the sales count. 

In [23]:
def remove_outliers(prices: pd.Series[Any] | list[float | int]) -> np.ndarray:
    """Replaces outlier-prices with the median of the non-outlier prices."""

    data = np.array(prices, dtype=float)
    # Identify outliers
    outliers = np.abs(data - np.mean(data)) > 3 * np.std(data)
    # Calculate the median of non-outliers (~ serves as a bit-wise version of "not") and replace outliers with it
    data[outliers] = np.median(data[~outliers])

    return data


def calc_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Computes the element-wise refund ratios given the sales count and refund count."""

    eps = 10**(-6)  # Prevents zero-division errors
    refund_ratio = refund_count / (sales_count + eps)
    # Since this expression is vectorized, refund_ratio will contain the element-wise results

    return refund_ratio


The functions were tested on several inputs, and it did work as intended.

## Task 4
The transactions are converted into a pandas DataFrame, with time as the index, and sorts them cronologically. We group by item and resampple into 1 hour windows to sum the price, build a pivot table of revenue per item across success, refund and cancelled. We also compute the refund ratio per item by using the function form task 3.

In [24]:
def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """Creates a pandas DataFrame from a list of transactions, indexed by 'timestamp' as datetime."""

    df = pd.DataFrame(transactions)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    # Sort it in case we receive and unsorted list of transactions
    df = df.sort_index()

    return df


def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Computes the hourly revenue for each item in the DataFrame."""

    return df.groupby('item').resample('1h')['price'].sum()


def status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a pivot table from the DataFrame with 'item' as the index, 'status' as the
    columns, and the sum of 'price' as the values.
    """

    return pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0
    )   # fill_value replaces missing values with 0


def refund_ratio_per_item(df: pd.DataFrame) -> pd.DataFrame:
    """Computes the refund ratio for each item in the DataFrame."""

    # Extracting the data
    pivot = status_pivot(df)
    sales = pivot['SUCCESS']
    refunds = pivot['REFUND']

    # Calculates the refund ratio (as a dataframe, so we could display the item names as well)
    return pd.DataFrame(
        {'refund ratio': calc_refund_ratio(np.array(sales.values), np.array(refunds.values))},
        index=pivot.index
    )


The functions were tested on several inputs, and it did work as intended.

## Task 5
Here, we use the previously defined function to solve several smaller tasks. Mainly, we wish to adjust the prices that are listed in NOK by scaling them by a factor of 10.0. \
Afterwards, we wish to create 4 output files, each containing different information about our data.

In [25]:
# The directory we wish for the code to search through
data_dir = 'data'

# Extract active register IDs and their stores
active_registers = metadata_reader(data_dir)
# Finding all log files (from active registers)
log_files = [
    path for path in file_finder(data_dir)
    if any(id in str(path) for id in active_registers)
]   # Making sure to only include logs from active registers

# Getting the transaction data as a dataframe, based on a list of dictionaries from each register
transactions = create_dataframe([log for path in log_files for log in log_parser(path)])
# Cleaning outliers
transactions["price"] = remove_outliers(transactions["price"])
# Correcting for currency by multiplying the price of every NOK transaction by 10
transactions.loc[transactions["currency"] == "NOK", "price"] *= 10.0
transactions["price"] = round(transactions["price"], 2)

# Writing files
transactions.to_csv("cleaned_transactions.csv")
hourly_revenue(transactions).to_json("hourly_revenue.json", indent=2)   # the keys are str(tuple[str, timestamp])
status_pivot(transactions).to_csv("status_pivot_table.csv")
refund_ratio_per_item(transactions).to_csv("item_refund_ratios.csv")


The resulting files were generated correctly inside the same folder as this notebook.
After analyzing them, these are our main take-aways:
1. 8am - 9am is by far the most lucrative time-slot, followed by 7am - 8am, and then 9am - 10am, where revenue drops significantly
2. The Espresso had a refund ratio of around 30%, making it the most refunded item (by approximately 5%)
3. The sandwich earned the most sales-revenue in total, out of the available menu items

These observations provide operational information that could prove benefitial in future profit-maximization ventures.